<a href="https://colab.research.google.com/github/akbar527/Akbar/blob/T1/2_RecA_PaDEL_Fingerprint_Modeling_Matrix_publication_ready.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RecA PaDEL Molecular Fingerprint and Modeling Matrix Workflow

This notebook prepares a publication-ready PaDEL fingerprint dataset for **RecA bioactivity classification**. It combines the clean step-by-step structure of the senior PaDEL notebook with the RecA-specific workflow from the user's project.

**Main purpose:** calculate molecular fingerprints using PaDEL-Descriptor and assemble a machine-learning-ready RecA modeling matrix.


## Publication-oriented workflow

1. Load the curated RecA bioactivity dataset generated from the previous data collection workflow.
2. Prepare a PaDEL-compatible `.smi` input file containing SMILES and ChEMBL compound IDs.
3. Calculate multiple binary PaDEL molecular fingerprints.
4. Merge all fingerprint types into one fingerprint matrix.
5. Merge the fingerprint matrix with the curated RecA bioactivity labels.
6. Export clean CSV files and summary metadata for downstream QSAR/ML modeling.

**Expected main input:**

```text
outputs/recA_chembl/recA_ec50_binary.csv
```

Optional balanced input for final modeling matrix:

```text
outputs/01_recA_ec50_binary_balanced_50_50.csv
```

**Main outputs:**

```text
outputs/recA_chembl/padel/molecule.smi
outputs/recA_chembl/padel/combined_fingerprints.csv
outputs/02_recA_modeling_matrix.csv
outputs/02_fingerprint_dataset_summary.csv
```


## Methodological notes

PaDEL fingerprints are calculated using `padelpy`, which provides a Python interface to PaDEL-Descriptor. The workflow keeps compound identifiers (`molecule_chembl_id`) traceable through the `Name` column generated by PaDEL.

For publication reporting, this notebook records the number of input compounds, successfully processed molecules, selected fingerprint families, final feature count, and class distribution.


In [1]:
# ============================================================
# 0. Optional installation cell
# ============================================================
# Run this cell only if padelpy is not installed in your environment.
# In Google Colab, uncomment the command below.

!pip install -q padelpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 68.5 MB/s eta 0:00:00


In [2]:
# ============================================================
# 1. Import libraries and configure paths
# ============================================================

from __future__ import annotations

import os
import glob
import shutil
import urllib.request
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

try:
    import padelpy
    from padelpy import padeldescriptor
except ImportError as exc:
    raise ImportError(
        "padelpy is not installed. Please run: pip install padelpy"
    ) from exc

# Notebook-safe project root. In notebooks, __file__ is usually unavailable.
PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs"
RECA_DIR = OUTPUT_DIR / "recA_chembl"
PADEL_DIR = RECA_DIR / "padel"
PADEL_DIR.mkdir(parents=True, exist_ok=True)

# Input from Notebook 1 / data collection stage
INPUT_FILE = RECA_DIR / "recA_ec50_binary.csv"

# Optional balanced data generated from the previous data curation stage
BALANCED_FILE = OUTPUT_DIR / "01_recA_ec50_binary_balanced_50_50.csv"

# PaDEL intermediate and final outputs
SMILES_FILE = PADEL_DIR / "molecule.smi"
PADEL_INPUT_FILE = PADEL_DIR / "recA_binary_padel_input.csv"
COMBINED_FP_FILE = PADEL_DIR / "combined_fingerprints.csv"

# Final machine-learning matrix outputs
FINAL_MATRIX_FILE = OUTPUT_DIR / "02_recA_modeling_matrix.csv"
SUMMARY_FILE = OUTPUT_DIR / "02_fingerprint_dataset_summary.csv"

print("Project directory:", PROJECT_DIR)
print("PaDEL output directory:", PADEL_DIR)


Project directory: /content
PaDEL output directory: /content/outputs/recA_chembl/padel


In [3]:
# ============================================================
# 2. Define fingerprint families
# ============================================================
# The senior notebook used these binary fingerprint families.
# Count-based fingerprints are intentionally excluded to keep the matrix binary and interpretable.

FINGERPRINT_TYPES = {
    "AtomPairs2D": "AtomPairs2DFingerprinter.xml",
    "EState": "EStateFingerprinter.xml",
    "CDKextended": "ExtendedFingerprinter.xml",
    "CDK": "Fingerprinter.xml",
    "CDKgraphonly": "GraphOnlyFingerprinter.xml",
    "KlekotaRoth": "KlekotaRothFingerprinter.xml",
    "MACCS": "MACCSFingerprinter.xml",
    "PubChem": "PubchemFingerprinter.xml",
    "Substructure": "SubstructureFingerprinter.xml",
}

list(FINGERPRINT_TYPES.items())


[('AtomPairs2D', 'AtomPairs2DFingerprinter.xml'),
 ('EState', 'EStateFingerprinter.xml'),
 ('CDKextended', 'ExtendedFingerprinter.xml'),
 ('CDK', 'Fingerprinter.xml'),
 ('CDKgraphonly', 'GraphOnlyFingerprinter.xml'),
 ('KlekotaRoth', 'KlekotaRothFingerprinter.xml'),
 ('MACCS', 'MACCSFingerprinter.xml'),
 ('PubChem', 'PubchemFingerprinter.xml'),
 ('Substructure', 'SubstructureFingerprinter.xml')]

In [4]:
# ============================================================
# 3. Fingerprint XML preparation
# ============================================================

def download_fingerprint_xml_files(destination: Path = PADEL_DIR) -> Path:
    """Download fingerprint XML files used by PaDEL if they are not already available."""
    destination.mkdir(parents=True, exist_ok=True)

    required_files = [destination / name for name in FINGERPRINT_TYPES.values()]
    if all(path.exists() for path in required_files):
        return destination

    zip_path = destination / "fingerprints_xml.zip"
    url = "https://github.com/dataprofessor/padel/raw/main/fingerprints_xml.zip"

    print("Downloading PaDEL fingerprint XML files...")
    urllib.request.urlretrieve(url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(destination)

    missing = [str(path.name) for path in required_files if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Missing fingerprint XML files after extraction: {missing}")

    return destination


def create_descriptor_xml_from_padelpy(fingerprint_name: str, descriptor_class_name: str) -> Path:
    """Create a single-fingerprint XML file directly from padelpy's descriptors.xml.

    This fallback is useful if external download is unavailable.
    """
    base_xml = Path(padelpy.__file__).parent / "PaDEL-Descriptor" / "descriptors.xml"
    if not base_xml.exists():
        raise FileNotFoundError(f"PaDEL descriptors.xml not found: {base_xml}")

    output_xml = PADEL_DIR / f"{fingerprint_name}.xml"
    tree = ET.parse(base_xml)
    root = tree.getroot()

    found = False
    for descriptor in root.iter("Descriptor"):
        is_target = descriptor.attrib.get("name") == descriptor_class_name
        descriptor.set("value", "true" if is_target else "false")
        if is_target:
            found = True

    if not found:
        raise ValueError(f"Descriptor class was not found in PaDEL descriptors.xml: {descriptor_class_name}")

    tree.write(output_xml, encoding="UTF-8", xml_declaration=True)
    return output_xml


def get_descriptor_xml(fingerprint_name: str, xml_filename: str) -> Path:
    """Return XML path for a fingerprint family, using downloaded XML when available."""
    xml_path = PADEL_DIR / xml_filename
    if xml_path.exists():
        return xml_path

    # Fallback mapping for padelpy descriptors.xml names
    fallback_names = {
        "AtomPairs2D": "AtomPairs2DFingerprinter",
        "EState": "EStateFingerprinter",
        "CDKextended": "ExtendedFingerprinter",
        "CDK": "Fingerprinter",
        "CDKgraphonly": "GraphOnlyFingerprinter",
        "KlekotaRoth": "KlekotaRothFingerprinter",
        "MACCS": "MACCSFingerprinter",
        "PubChem": "PubchemFingerprinter",
        "Substructure": "SubstructureFingerprinter",
    }
    return create_descriptor_xml_from_padelpy(fingerprint_name, fallback_names[fingerprint_name])

try:
    download_fingerprint_xml_files(PADEL_DIR)
except Exception as err:
    print("XML download was not completed. The workflow will use padelpy descriptors.xml fallback.")
    print("Reason:", err)

xml_files = sorted(str(path.name) for path in PADEL_DIR.glob("*.xml"))
xml_files[:10], len(xml_files)


(['AtomPairs2DFingerprintCount.xml',
  'AtomPairs2DFingerprinter.xml',
  'EStateFingerprinter.xml',
  'ExtendedFingerprinter.xml',
  'Fingerprinter.xml',
  'GraphOnlyFingerprinter.xml',
  'KlekotaRothFingerprintCount.xml',
  'KlekotaRothFingerprinter.xml',
  'MACCSFingerprinter.xml',
  'PubchemFingerprinter.xml'],
 12)

In [5]:

# ============================================================
# 4. Load and validate RecA dataset
# ============================================================

def load_reca_dataset(input_file: Path = INPUT_FILE, limit: int | None = None) -> pd.DataFrame:
    """Load curated RecA binary dataset from the previous data collection workflow."""
    if not input_file.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_file}\n"
            "Please run Notebook 1 first and make sure recA_ec50_binary.csv exists."
        )

    df = pd.read_csv(input_file)

    # Accept common column variants but standardize them for this workflow.
    rename_map = {}
    if "canonical_smiles" not in df.columns and "smiles" in df.columns:
        rename_map["smiles"] = "canonical_smiles"
    if "molecule_chembl_id" not in df.columns and "Name" in df.columns:
        rename_map["Name"] = "molecule_chembl_id"
    if rename_map:
        df = df.rename(columns=rename_map)

    required = {"molecule_chembl_id", "canonical_smiles"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    if "bioactivity_class" not in df.columns and "class" in df.columns:
        # If class is numeric, recover text labels for reporting.
        df["bioactivity_class"] = df["class"].map({1: "active", 0: "inactive"}).fillna(df["class"].astype(str))

    if "class" not in df.columns:
        if "bioactivity_numeric" in df.columns:
            df["class"] = df["bioactivity_numeric"]
        elif "bioactivity_class" in df.columns:
            df["class"] = df["bioactivity_class"].str.lower().map({"active": 1, "inactive": 0})

    if "class" not in df.columns:
        raise ValueError("No usable class label found. Expected class, bioactivity_numeric, or bioactivity_class.")

    df = df.dropna(subset=["molecule_chembl_id", "canonical_smiles", "class"]).copy()
    df["molecule_chembl_id"] = df["molecule_chembl_id"].astype(str)
    df["canonical_smiles"] = df["canonical_smiles"].astype(str)
    df["class"] = df["class"].astype(int)

    # Remove duplicate molecules by ChEMBL ID to avoid merge ambiguity.
    before = len(df)
    df = df.drop_duplicates(subset=["molecule_chembl_id"], keep="first").copy()
    after = len(df)
    if before != after:
        print(f"Removed {before - after} duplicate molecule_chembl_id rows.")

    if limit is not None:
        df = df.head(limit).copy()

    return df

# Optional quick test: set LIMIT = 10 for trial run, or None for full dataset.
LIMIT = None

# The file recA_ec50_binary.csv was found at /content/recA_ec50_binary.csv,
# but the default INPUT_FILE path points to /content/outputs/recA_chembl/recA_ec50_binary.csv.
# We explicitly pass the correct file path to the function.
df = load_reca_dataset(input_file=Path("/content/recA_ec50_binary.csv"), limit=LIMIT)
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
print("Class distribution:")
print(df["class"].value_counts().sort_index())
df.head()


Dataset shape: (915, 7)
Columns: ['molecule_chembl_id', 'canonical_smiles', 'standard_value', 'standard_value_nM', 'pEC50', 'bioactivity_class', 'class']
Class distribution:
class
0    601
1    314
Name: count, dtype: int64


,molecule_chembl_id,canonical_smiles,standard_value,standard_value_nM,pEC50,bioactivity_class,class
0,CHEMBL1463659,Cn1nc(-c2ccccn2)nc2c(=O)n(C)c(=O)nc1-2,19.1,19.1,7.718967,active,1
1,CHEMBL3195749,COC(=O)c1ccccc1NC1=C/C(=N\S(=O)(=O)c2cccs2)c2c...,20.8,20.8,7.681937,active,1
2,CHEMBL601757,Cc1nc2c(=O)n(C)c(=O)nc-2n(C)n1,20.9,20.9,7.679854,active,1
3,CHEMBL1334062,Cn1nc(-c2ccc(Cl)cc2)nc2c(=O)n(C)c(=O)nc1-2,30.5,30.5,7.515700,active,1
4,CHEMBL1370184,CC(=O)Oc1c(/C=C(\C#N)C(N)=O)[nH]c2ccccc12,31.4,31.4,7.503070,active,1


In [6]:
# ============================================================
# 5. Prepare PaDEL .smi input file
# ============================================================

def prepare_padel_input(df: pd.DataFrame) -> pd.DataFrame:
    """Create PaDEL-compatible molecule.smi and save metadata table."""
    subset_cols = ["molecule_chembl_id", "canonical_smiles", "class"]
    if "bioactivity_class" in df.columns:
        subset_cols.insert(2, "bioactivity_class")

    subset = df[subset_cols].copy()
    subset.to_csv(PADEL_INPUT_FILE, index=False)

    # PaDEL requires: SMILES<TAB>Name without header.
    smiles_table = subset[["canonical_smiles", "molecule_chembl_id"]]
    smiles_table.to_csv(SMILES_FILE, sep="\t", index=False, header=False)

    return subset

padel_input = prepare_padel_input(df)
print("PaDEL input file saved to:", PADEL_INPUT_FILE)
print("SMILES file saved to:", SMILES_FILE)
padel_input.head()


PaDEL input file saved to: /content/outputs/recA_chembl/padel/recA_binary_padel_input.csv
SMILES file saved to: /content/outputs/recA_chembl/padel/molecule.smi


,molecule_chembl_id,canonical_smiles,bioactivity_class,class
0,CHEMBL1463659,Cn1nc(-c2ccccn2)nc2c(=O)n(C)c(=O)nc1-2,active,1
1,CHEMBL3195749,COC(=O)c1ccccc1NC1=C/C(=N\S(=O)(=O)c2cccs2)c2c...,active,1
2,CHEMBL601757,Cc1nc2c(=O)n(C)c(=O)nc-2n(C)n1,active,1
3,CHEMBL1334062,Cn1nc(-c2ccc(Cl)cc2)nc2c(=O)n(C)c(=O)nc1-2,active,1
4,CHEMBL1370184,CC(=O)Oc1c(/C=C(\C#N)C(N)=O)[nH]c2ccccc12,active,1


In [7]:
# ============================================================
# 6. Calculate PaDEL fingerprints
# ============================================================

def calculate_fingerprint_family(
    fingerprint_name: str,
    xml_filename: str,
    threads: int = 2,
    timeout: int | None = None,
) -> pd.DataFrame:
    """Calculate one PaDEL fingerprint family and return its DataFrame."""
    descriptor_xml = get_descriptor_xml(fingerprint_name, xml_filename)
    output_file = PADEL_DIR / f"{fingerprint_name}.csv"

    print(f"Calculating {fingerprint_name} fingerprints...")
    padel_args = {
        "mol_dir": str(SMILES_FILE),
        "d_file": str(output_file),
        "descriptortypes": str(descriptor_xml),
        "detectaromaticity": True,
        "standardizenitro": True,
        "standardizetautomers": True,
        "removesalt": True,
        "fingerprints": True,
        "threads": threads,
        "log": True,
    }
    if timeout is not None:
        padel_args["maxruntime"] = timeout

    padeldescriptor(**padel_args)

    if not output_file.exists():
        raise FileNotFoundError(f"PaDEL did not create output file: {output_file}")

    desc = pd.read_csv(output_file)
    if "Name" not in desc.columns:
        raise ValueError(f"The {fingerprint_name} output does not contain a Name column.")

    # Avoid duplicated columns when merging different fingerprint families.
    renamed_cols = {
        col: f"{fingerprint_name}_{col}"
        for col in desc.columns
        if col != "Name" and not col.startswith(f"{fingerprint_name}_")
    }
    desc = desc.rename(columns=renamed_cols)

    return desc


def calculate_all_fingerprints(threads: int = 2, timeout: int | None = None) -> pd.DataFrame:
    """Calculate and merge all selected fingerprint families."""
    merged = None

    for fp_name, xml_file in FINGERPRINT_TYPES.items():
        desc = calculate_fingerprint_family(fp_name, xml_file, threads=threads, timeout=timeout)
        merged = desc if merged is None else merged.merge(desc, on="Name", how="outer", validate="one_to_one")

    if merged is None or merged.empty:
        raise ValueError("No fingerprint descriptors were generated.")

    # PaDEL should produce binary features. Fill rare missing values with 0 after outer merge.
    feature_cols = [c for c in merged.columns if c != "Name"]
    merged[feature_cols] = merged[feature_cols].fillna(0)

    return merged

# Runtime settings
THREADS = 2
TIMEOUT = None

all_fingerprints = calculate_all_fingerprints(threads=THREADS, timeout=TIMEOUT)
print("Fingerprint matrix shape:", all_fingerprints.shape)
all_fingerprints.head()


Calculating AtomPairs2D fingerprints...
Calculating EState fingerprints...
Calculating CDKextended fingerprints...
Calculating CDK fingerprints...
Calculating CDKgraphonly fingerprints...
Calculating KlekotaRoth fingerprints...
Calculating MACCS fingerprints...
Calculating PubChem fingerprints...
Calculating Substructure fingerprints...
Fingerprint matrix shape: (915, 10146)


,Name,AtomPairs2D_AD2D1,AtomPairs2D_AD2D2,AtomPairs2D_AD2D3,AtomPairs2D_AD2D4,AtomPairs2D_AD2D5,AtomPairs2D_AD2D6,AtomPairs2D_AD2D7,AtomPairs2D_AD2D8,AtomPairs2D_AD2D9,...,Substructure_SubFP298,Substructure_SubFP299,Substructure_SubFP300,Substructure_SubFP301,Substructure_SubFP302,Substructure_SubFP303,Substructure_SubFP304,Substructure_SubFP305,Substructure_SubFP306,Substructure_SubFP307
0,CHEMBL1077990,1,1,1,1,0,0,0,0,0,...,0,0,1,1,1,1,0,0,0,1
1,CHEMBL1165723,1,1,1,0,0,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
2,CHEMBL1200938,1,1,1,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
3,CHEMBL1201049,1,1,1,0,0,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
4,CHEMBL1256974,1,1,1,1,0,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1


In [8]:
# ============================================================
# 7. Build combined fingerprint table with RecA labels
# ============================================================

def build_combined_fingerprint_dataset(
    fingerprints: pd.DataFrame,
    metadata: pd.DataFrame,
) -> pd.DataFrame:
    """Merge PaDEL fingerprints with RecA compound labels."""
    meta = metadata.copy()
    meta["Name"] = meta["molecule_chembl_id"].astype(str)

    combined = meta.merge(
        fingerprints,
        on="Name",
        how="inner",
        validate="one_to_one",
    )

    if combined.empty:
        raise ValueError("No compounds matched between PaDEL fingerprints and RecA metadata.")

    # Reorder key columns for readability.
    key_cols = ["Name", "molecule_chembl_id", "canonical_smiles"]
    if "bioactivity_class" in combined.columns:
        key_cols.append("bioactivity_class")
    key_cols.append("class")

    other_cols = [c for c in combined.columns if c not in key_cols]
    combined = combined[key_cols + other_cols]

    combined.to_csv(COMBINED_FP_FILE, index=False)
    return combined

combined_fingerprints = build_combined_fingerprint_dataset(all_fingerprints, padel_input)
print("Combined fingerprint dataset saved to:", COMBINED_FP_FILE)
print("Combined shape:", combined_fingerprints.shape)
combined_fingerprints.head()


Combined fingerprint dataset saved to: /content/outputs/recA_chembl/padel/combined_fingerprints.csv
Combined shape: (915, 10150)


,Name,molecule_chembl_id,canonical_smiles,bioactivity_class,class,AtomPairs2D_AD2D1,AtomPairs2D_AD2D2,AtomPairs2D_AD2D3,AtomPairs2D_AD2D4,AtomPairs2D_AD2D5,...,Substructure_SubFP298,Substructure_SubFP299,Substructure_SubFP300,Substructure_SubFP301,Substructure_SubFP302,Substructure_SubFP303,Substructure_SubFP304,Substructure_SubFP305,Substructure_SubFP306,Substructure_SubFP307
0,CHEMBL1463659,CHEMBL1463659,Cn1nc(-c2ccccn2)nc2c(=O)n(C)c(=O)nc1-2,active,1,1,1,1,0,0,...,0,0,1,1,0,0,0,0,0,1
1,CHEMBL3195749,CHEMBL3195749,COC(=O)c1ccccc1NC1=C/C(=N\S(=O)(=O)c2cccs2)c2c...,active,1,1,1,1,1,0,...,0,0,1,1,1,0,0,0,0,1
2,CHEMBL601757,CHEMBL601757,Cc1nc2c(=O)n(C)c(=O)nc-2n(C)n1,active,1,1,1,1,0,0,...,0,0,1,1,0,0,0,0,0,1
3,CHEMBL1334062,CHEMBL1334062,Cn1nc(-c2ccc(Cl)cc2)nc2c(=O)n(C)c(=O)nc1-2,active,1,1,1,1,0,0,...,0,0,1,1,0,0,0,0,0,1
4,CHEMBL1370184,CHEMBL1370184,CC(=O)Oc1c(/C=C(\C#N)C(N)=O)[nH]c2ccccc12,active,1,1,1,1,0,0,...,0,0,1,1,1,1,0,0,0,1


In [9]:
# ============================================================
# 8. Assemble final modeling matrix
# ============================================================

# Redefine INPUT_FILE and BALANCED_FILE to their actual locations in /content
# This temporary redefinition is necessary because the global variables were set
# to a nested path that does not exist for these files in the current environment.
INPUT_FILE = Path("/content/recA_ec50_binary.csv")
BALANCED_FILE = Path("/content/recA_ec50_binary_balanced_50_50.csv")

def load_modeling_base() -> pd.DataFrame:
    """Use the balanced dataset if available; otherwise use the curated input dataset."""
    if BALANCED_FILE.exists():
        print("Using balanced dataset:", BALANCED_FILE)
        base = pd.read_csv(BALANCED_FILE)
    else:
        print("Balanced dataset was not found. Using curated RecA dataset:", INPUT_FILE)
        base = pd.read_csv(INPUT_FILE)

    if "molecule_chembl_id" not in base.columns and "Name" in base.columns:
        base = base.rename(columns={"Name": "molecule_chembl_id"})
    if "canonical_smiles" not in base.columns and "smiles" in base.columns:
        base = base.rename(columns={"smiles": "canonical_smiles"})
    if "class" not in base.columns:
        if "bioactivity_numeric" in base.columns:
            base["class"] = base["bioactivity_numeric"]
        elif "bioactivity_class" in base.columns:
            base["class"] = base["bioactivity_class"].str.lower().map({"active": 1, "inactive": 0})

    base = base.dropna(subset=["molecule_chembl_id", "class"]).copy()
    base["molecule_chembl_id"] = base["molecule_chembl_id"].astype(str)
    base["class"] = base["class"].astype(int)
    base = base.drop_duplicates(subset=["molecule_chembl_id"], keep="first")
    return base


def assemble_modeling_matrix(base: pd.DataFrame, combined_fp: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Create final ML-ready RecA fingerprint matrix."""
    fp_meta_cols = {
        "Name", "molecule_chembl_id", "canonical_smiles", "bioactivity_class", "class",
        "bioactivity_numeric", "standard_value", "standard_units", "standard_type",
    }
    fp_feature_cols = [c for c in combined_fp.columns if c not in fp_meta_cols]

    fp_features = combined_fp[["Name"] + fp_feature_cols].copy()

    modeling = base.merge(
        fp_features,
        left_on="molecule_chembl_id",
        right_on="Name",
        how="inner",
        validate="one_to_one",
    )

    if modeling.empty:
        raise ValueError("Final matrix is empty. Check molecule_chembl_id / Name matching.")

    # Keep publication-friendly column order.
    first_cols = []
    for col in ["molecule_chembl_id", "Name", "canonical_smiles", "bioactivity_class", "class"]:
        if col in modeling.columns:
            first_cols.append(col)
    remaining_cols = [c for c in modeling.columns if c not in first_cols]
    modeling = modeling[first_cols + remaining_cols]

    # Replace any remaining missing fingerprint values with 0.
    modeling[fp_feature_cols] = modeling[fp_feature_cols].fillna(0)

    modeling.to_csv(FINAL_MATRIX_FILE, index=False)
    return modeling, fp_feature_cols

base_dataset = load_modeling_base()
final_matrix, fingerprint_feature_cols = assemble_modeling_matrix(base_dataset, combined_fingerprints)

print("Final modeling matrix saved to:", FINAL_MATRIX_FILE)
print("Final matrix shape:", final_matrix.shape)
print("Number of fingerprint features:", len(fingerprint_feature_cols))
print("Class distribution:")
print(final_matrix["class"].value_counts().sort_index())
final_matrix.head()


Using balanced dataset: /content/recA_ec50_binary_balanced_50_50.csv
Final modeling matrix saved to: /content/outputs/02_recA_modeling_matrix.csv
Final matrix shape: (628, 10153)
Number of fingerprint features: 10145
Class distribution:
class
0    314
1    314
Name: count, dtype: int64


,molecule_chembl_id,Name,canonical_smiles,bioactivity_class,class,standard_value,standard_value_nM,pEC50,AtomPairs2D_AD2D1,AtomPairs2D_AD2D2,...,Substructure_SubFP298,Substructure_SubFP299,Substructure_SubFP300,Substructure_SubFP301,Substructure_SubFP302,Substructure_SubFP303,Substructure_SubFP304,Substructure_SubFP305,Substructure_SubFP306,Substructure_SubFP307
0,CHEMBL1341981,CHEMBL1341981,Cc1ccccc1CS(=O)(=O)c1nnc([C@H](CC(C)C)NC(=O)OC...,inactive,0,11600.0,11600.0,4.935542,1,1,...,0,0,1,1,1,0,0,0,0,1
1,CHEMBL1595992,CHEMBL1595992,COC(=O)c1n[nH]c(NC(=O)c2ccc(C(C)(C)C)cc2)n1,inactive,0,10100.0,10100.0,4.995679,1,1,...,0,0,1,1,1,0,0,0,0,1
2,CHEMBL1511042,CHEMBL1511042,FC(F)(F)c1ccc(C2[C@H]3CN(Cc4cccnc4)[C@H](c4ccc...,inactive,0,24100.0,24100.0,4.617983,1,1,...,0,0,1,1,1,0,0,0,0,1
3,CHEMBL1590201,CHEMBL1590201,Cc1c([N+](=O)[O-])cnc(Sc2ccccc2)c1[N+](=O)[O-],active,1,47.9,47.9,7.319664,1,1,...,1,1,1,1,1,0,0,0,0,1
4,CHEMBL1500823,CHEMBL1500823,O=C(CSc1ncnc2c1nnn2Cc1ccccc1F)NCC1CCCO1,inactive,0,16500.0,16500.0,4.782516,1,1,...,0,0,1,1,1,0,0,0,0,1


In [10]:
# ============================================================
# 9. Save reproducibility summary
# ============================================================

summary = pd.DataFrame({
    "item": [
        "target_protein",
        "activity_endpoint",
        "input_file",
        "balanced_file_used",
        "fingerprint_families",
        "input_compounds",
        "processed_compounds",
        "final_modeling_rows",
        "fingerprint_feature_count",
        "active_class_count",
        "inactive_class_count",
        "combined_fingerprint_output",
        "final_modeling_matrix_output",
    ],
    "value": [
        "RecA",
        "EC50-derived binary classification",
        str(INPUT_FILE),
        str(BALANCED_FILE.exists()),
        ", ".join(FINGERPRINT_TYPES.keys()),
        len(df),
        len(combined_fingerprints),
        len(final_matrix),
        len(fingerprint_feature_cols),
        int((final_matrix["class"] == 1).sum()),
        int((final_matrix["class"] == 0).sum()),
        str(COMBINED_FP_FILE),
        str(FINAL_MATRIX_FILE),
    ]
})

summary.to_csv(SUMMARY_FILE, index=False)
print("Summary saved to:", SUMMARY_FILE)
summary


Summary saved to: /content/outputs/02_fingerprint_dataset_summary.csv


,item,value
0,target_protein,RecA
1,activity_endpoint,EC50-derived binary classification
2,input_file,/content/recA_ec50_binary.csv
3,balanced_file_used,True
4,fingerprint_families,"AtomPairs2D, EState, CDKextended, CDK, CDKgrap..."
5,input_compounds,915
6,processed_compounds,915
7,final_modeling_rows,628
8,fingerprint_feature_count,10145
9,active_class_count,314


## Final output for downstream machine learning

The main file for the next workflow is:

```text
outputs/02_recA_modeling_matrix.csv
```

This file contains compound identifiers, SMILES, RecA bioactivity labels, and PaDEL fingerprint features. It is ready for feature selection, model construction, hyperparameter tuning, and external/FDA drug prediction workflows.

For manuscript methods, this notebook supports a concise statement such as:

> Molecular fingerprints were calculated from canonical SMILES using PaDEL-Descriptor through the `padelpy` interface. Nine binary fingerprint families were generated, including AtomPairs2D, EState, CDK, CDK extended, CDK graph-only, Klekota-Roth, MACCS, PubChem, and Substructure fingerprints. The generated fingerprint matrix was merged with the curated RecA bioactivity labels using ChEMBL compound identifiers to construct the final QSAR modeling matrix.
